In [1]:
# imports
import os
import time
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.metrics import auc

import utils as ut

In [2]:
fs = 8000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds
tolerance = 5  # Hz
nperseg = 1000
nfft = None # 2*nperseg
noverlap = 0.5
window = 'hanning'
dc = 20
crop_freq = 8000
norm_size = 9
modes = ["wasserstein", "edge_count", "laplacian"]
quantization_level = 3
raw_signal = ut.simulate_raw_signal(f0, fs, duration)

wasserstein_detector = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_level, mode="wasserstein")
edge_count_detector = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_level, mode="edge_count")
laplacian_detector = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_level, mode="laplacian")

In [3]:
snr = 10
rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')
F, T, Sxx, phasogram = ut.calc_spectrogram(x=rx, fs=fs, nperseg=nperseg, percent_overlap=noverlap, nfft=nfft, window=window, remove_dc=dc, crop_freq=crop_freq, logscale=True)

K_wasserstein, F_wasserstein = wasserstein_detector.get_feature_vector(rx)
K_edge_count, F_edge_count = edge_count_detector.get_feature_vector(rx)
K_laplacian, F_laplacian = laplacian_detector.get_feature_vector(rx)

fig = make_subplots(rows=1, cols=4, column_titles=["Spectrogram", "Edge Count", "Laplacian", "Wasserstein"], column_widths=[0.7, 0.1, 0.1, 0.1])
fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='Viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=K_edge_count, y=F_edge_count, mode='lines', marker=dict(color='blue'), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=K_laplacian, y=F_laplacian, mode='lines', marker=dict(color='green'), showlegend=False), row=1, col=3)
fig.add_trace(go.Scatter(x=K_wasserstein, y=F_wasserstein, mode='lines', marker=dict(color='red'), showlegend=False), row=1, col=4)
fig.update_layout(height=600, width=1200)
fig.show()

In [5]:
snr = -10
rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')
F, T, Sxx, phasogram = ut.calc_spectrogram(x=rx, fs=fs, nperseg=nperseg, percent_overlap=noverlap, nfft=nfft, window=window, remove_dc=dc, crop_freq=crop_freq, logscale=True)

K_wasserstein, F_wasserstein = wasserstein_detector.get_feature_vector(rx)
K_edge_count, F_edge_count = edge_count_detector.get_feature_vector(rx)
K_laplacian, F_laplacian = laplacian_detector.get_feature_vector(rx)

fig = make_subplots(rows=1, cols=4, column_titles=["Spectrogram", "Edge Count", "Laplacian", "Wasserstein"], column_widths=[0.7, 0.1, 0.1, 0.1])
fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='Viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=K_edge_count, y=F_edge_count, mode='lines', marker=dict(color='blue'), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=K_laplacian, y=F_laplacian, mode='lines', marker=dict(color='green'), showlegend=False), row=1, col=3)
fig.add_trace(go.Scatter(x=K_wasserstein, y=F_wasserstein, mode='lines', marker=dict(color='red'), showlegend=False), row=1, col=4)
fig.update_layout(height=600, width=1200)
fig.show()